In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os


## Load datasets

In [ ]:
path = '/content/drive/MyDrive/cold_heads_monitoring/Dataset/'
arrays = []

for filename in os.listdir(path):
    print(filename)
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:

            arr = np.asarray(pickle.load(f))[:, 2:3, :]
            arrays.append(arr)


## Balance datasets

In [ ]:
import sys

min_len = 2500

balanced_arrays = [arr[:min_len] for arr in arrays]

head_new = np.squeeze(np.asarray(balanced_arrays[0]),1)
dataset_no_comp = np.squeeze(np.asarray(balanced_arrays[1]),1)

head_30k_ore = np.squeeze(np.asarray(balanced_arrays[2]),1)

head_new.shape, dataset_no_comp.shape, head_30k_ore.shape


In [ ]:
data = np.concatenate([head_new,dataset_no_comp,head_30k_ore], axis=0)
np.random.shuffle(data)
data.shape


## Training DBSCAN

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

class IncrementalDBSCAN:
    def __init__(self, epsilon, min_samples):
        self.epsilon = epsilon
        self.min_samples = min_samples
        self.dbscan = None

    def fit(self, X):
        self.dbscan = DBSCAN(eps=self.epsilon, min_samples=self.min_samples)
        self.dbscan.fit(X)
        dim  = len(self.dbscan.components_)
        plt.scatter(self.dbscan.components_[:, 0], self.dbscan.components_[:, 1], c=self.dbscan.labels_[0:dim], cmap='viridis')
        plt.show()

    def incremental_dbscan(self,dbscan_model, new_data, eps, min_samples):

      components = dbscan_model.components_
      core_samples = dbscan_model.core_sample_indices_

      updated_components = np.concatenate((components, new_data))
      updated_core_samples = np.concatenate((core_samples, np.arange(len(components), len(components) + len(new_data))))

      updated_dbscan = DBSCAN(eps=eps, min_samples=min_samples)
      updated_dbscan.components_ = updated_components
      updated_dbscan.core_sample_indices_ = updated_core_samples

      updated_labels = updated_dbscan.fit_predict(updated_components)

      plt.scatter(updated_dbscan.components_[:, 0], updated_dbscan.components_[:, 1], c=updated_labels[:], cmap='viridis')
      plt.show()

      return updated_labels


In [ ]:
dbscan = IncrementalDBSCAN(1, 10)
dbscan.fit(np.concatenate([head_30k_ore,dataset_no_comp], axis=0))

initial_labels = dbscan.dbscan.labels_
n_clusters = len(set(initial_labels)) - (1 if -1 in initial_labels else 0)
print("Number of clusters:", n_clusters)

n_anomalies = list(initial_labels).count(-1)
print("Number of noise points:", n_anomalies)


In [ ]:
new_data = head_new[0:1500]
updated_labels = dbscan.incremental_dbscan(dbscan.dbscan,new_data,1,5)

n_clusters_2 = len(set(updated_labels)) - (1 if -1 in updated_labels else 0)
print("Number of clusters after update:", n_clusters_2)

n_anomalies_2 = list(updated_labels).count(-1)
print("Number of noise points after update:", n_anomalies_2)
